# DBCenter in Colab: a real PostgreSQL in one `pip install`

This notebook starts an actual PostgreSQL 18 inside the Colab VM - no server, no Docker, no account - and uses it the way an AI agent would: plain SQL, a read-only mode PostgreSQL itself enforces, vector search for memory, and a copy of the data in a bucket.

[GitHub](https://github.com/heyobi/dbcenter) · [PyPI](https://pypi.org/project/dbcenter/)

In [ ]:
!pip install -q "dbcenter[mcp]"

## One Colab-specific step

PostgreSQL refuses to run as root, and Colab runs everything as root. So we create an ordinary user and run each example as a small script under that user. Outside Colab - on a laptop, a server, a normal container user - none of this is needed: you just `import dbcenter`.

In [ ]:
!id -u dbuser >/dev/null 2>&1 || useradd -m dbuser
!mkdir -p /home/dbuser/work && chown -R dbuser:dbuser /home/dbuser

## 1. A database, with SQL

`DBCenter(path=...)` starts PostgreSQL against a data directory; the data survives restarts. The first run downloads the PostgreSQL binaries (about 60 MB) and caches them.

In [ ]:
%%writefile /home/dbuser/work/sql.py
import dbcenter

db = dbcenter.DBCenter(path="/home/dbuser/work/data")
db.query("CREATE TABLE IF NOT EXISTS orders (id serial PRIMARY KEY, customer text, total numeric)")
db.query("INSERT INTO orders (customer, total) VALUES ($1, $2), ($3, $4), ($5, $6)",
         ["Ali", 10.5, "Ayşe", 99, "Ali", 42])
print(db.query("SELECT customer, count(*) AS orders, sum(total) AS spent "
               "FROM orders GROUP BY customer ORDER BY spent DESC"))
db.shutdown()

In [ ]:
!cd /home/dbuser/work && sudo -u dbuser python3 sql.py

## 2. SQL an agent cannot write through

`read_only=True` runs the statement in a READ ONLY transaction that is rolled back. PostgreSQL refuses any write - including ones a keyword filter would miss, like a `DELETE` inside a CTE.

In [ ]:
%%writefile /home/dbuser/work/agent_sql.py
import dbcenter

db = dbcenter.DBCenter(path="/home/dbuser/work/data")
print(db.query("SELECT sum(total) AS revenue FROM orders", read_only=True))

for attack in ["DELETE FROM orders",
               "WITH gone AS (DELETE FROM orders RETURNING *) SELECT count(*) FROM gone",
               "DROP TABLE orders"]:
    try:
        db.query(attack, read_only=True)
        print("NOT refused:", attack)
    except Exception as e:
        print("refused:", attack)

print("orders still there:", db.query("SELECT count(*) AS n FROM orders")[0]["n"])
db.shutdown()

In [ ]:
!cd /home/dbuser/work && sudo -u dbuser python3 agent_sql.py

The same guarantee is available as an MCP server for Claude, Cursor or any MCP client: `dbcenter-mcp --path /absolute/path/to/data` (read-only unless started with `--allow-writes`).

## 3. Memory for an agent: pgvector

Extensions install with one call. Real embeddings would come from a model; three-dimensional vectors keep the example short.

In [ ]:
%%writefile /home/dbuser/work/memory.py
import dbcenter

db = dbcenter.DBCenter(path="/home/dbuser/work/data")
db.install_extension("pgvector")
db.query("CREATE TABLE IF NOT EXISTS memories (id serial PRIMARY KEY, text text, embedding vector(3))")
db.query("TRUNCATE memories")
db.query("INSERT INTO memories (text, embedding) VALUES "
         "('the user likes tea', '[1,0,0]'), "
         "('the user lives in Ankara', '[0,1,0]'), "
         "('the user has a cat', '[0,0,1]')")
print(db.query("SELECT text FROM memories ORDER BY embedding <-> '[0.9,0.2,0]' LIMIT 2"))
db.shutdown()

In [ ]:
!cd /home/dbuser/work && sudo -u dbuser python3 memory.py

## 4. A copy in a bucket, and a restore from it

With `multi_user_sync=True`, each table ships what changed since the last sync to storage as Parquet. Here the bucket is a folder; with `dbcenter.S3StorageAdapter(...)` it is S3, Cloudflare R2 or MinIO. A second database - another machine, or this one after losing its disk - rebuilds from it.

In [ ]:
%%writefile /home/dbuser/work/bucket.py
import dbcenter

bucket = dbcenter.LocalStorageAdapter("/home/dbuser/work/bucket")

laptop = dbcenter.DBCenter(path="/home/dbuser/work/laptop", multi_user_sync=True)
laptop.collection("notes").insert_many([{"body": "remember the milk"}, {"body": "call Ayşe"}])
print("synced:", dbcenter.SyncEngine(laptop, bucket).sync_table("notes", 10_000))
laptop.shutdown()

server = dbcenter.DBCenter(path="/home/dbuser/work/server", multi_user_sync=True)
dbcenter.SyncEngine(server, bucket).pull_and_apply("notes")
print("restored:", server.query("SELECT body FROM notes ORDER BY body"))
server.shutdown()

In [ ]:
!cd /home/dbuser/work && sudo -u dbuser python3 bucket.py
!ls -la /home/dbuser/work/bucket

## Where next

- [README](https://github.com/heyobi/dbcenter#readme): the API, the bucket, columnar reads, the container.
- [docs/mcp.md](https://github.com/heyobi/dbcenter/blob/main/docs/mcp.md): the MCP server.
- [docs/cost-and-scale.md](https://github.com/heyobi/dbcenter/blob/main/docs/cost-and-scale.md): what it costs and how fast it is, measured, including where it is slow.

Issues and questions: https://github.com/heyobi/dbcenter/issues